# Comparación estadística final de los diez mejores modelos

**experiment027 — Inferencia pareada de exhaustividad (recall)**

Se recuperan las predicciones out-of-fold (OOF) de la corrida experiment024 documentada en experiment026, sin ajustar ningún modelo ni utilizar las etiquetas de prueba. Los diez finalistas se ordenan por ROC AUC, como en el notebook principal. Se comparan los 45 pares no ordenados.

**Análisis principal:** política de alerta con objetivo de recall 98 %, umbral calculado en los otros cuatro folds (misma implementación del notebook final). **Sensibilidad:** umbral fijo 0,50 para todos. El contraste principal es McNemar exacto bilateral, con Holm sobre los 45 pares; Bonferroni simple y bootstrap centrado se reportan también. La sensibilidad constituye una familia exploratoria separada; no se combinan afirmaciones de las dos políticas sin un ajuste conjunto.

Los resultados son condicionales y exploratorios: los modelos fueron elegidos usando estas OOF, los folds de ajuste se solapan y las transacciones podrían ser dependientes. No son inferencia confirmatoria sobre modelos reentrenados ni sobre un conjunto externo.


In [ ]:
from pathlib import Path
import sys, json, platform
import numpy as np
import pandas as pd
import scipy, sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/experiment026/resultados/top10.csv').is_file()),None)
if ROOT is None: raise FileNotFoundError('Abrir Jupyter en la raíz de Credit-Fraud.')
sys.path.insert(0,str(ROOT/'experiments/experiment027'))
from recall_inference import read_labels,read_oof,decisions,pairwise_recall,p_matrix

CHECKPOINTS=ROOT/'outputs/experiment024'  # Cambiar si se guardaron en otro directorio.
REFERENCE=ROOT/'experiments/experiment026/resultados'
OUT=ROOT/'outputs/experiment027'
OUT.mkdir(parents=True,exist_ok=True)
B=100_000
SEED=20261125
ALPHA=.05
TARGET=.98
sns.set_theme(style='white',context='notebook')


## 1. Recuperar los finalistas y verificar la corrida

Se leen exclusivamente las predicciones OOF guardadas por fold. Se comprueban la firma de la corrida, los índices de validación reconstruidos con semilla 42, la cobertura, el rango de probabilidades y las cuatro métricas publicadas (AUC, AP, log-loss y Brier). Se requiere el orden original del CSV de entrenamiento. La firma almacenada identifica la corrida; no se recalcula a partir de todos los datos y no prueba por sí sola la identidad de cada fila.


In [ ]:
ranking=pd.read_csv(REFERENCE/'top10.csv')
manifest=json.loads((REFERENCE/'resumen_final.json').read_text(encoding='utf8'))
y=read_labels(ROOT)
P,folds,audit=read_oof(CHECKPOINTS,ranking,manifest,y)
names=[f'M{i:02}' for i in ranking.puesto]
ranking.insert(0,'modelo',names)
display(ranking[['modelo','familia','roc_auc','average_precision']])
display(audit)
print(f'{len(y):,} transacciones | {int(y.sum()):,} fraudes | {len(names)} modelos')
ranking.to_csv(OUT/'modelos_top10.csv',index=False)
audit.to_csv(OUT/'auditoria_oof.csv',index=False)


## 2. Definición del estimando y políticas de alerta

Sobre los $n_+$ fraudes observados, $Z_{im}=\mathbf{1}\{\hat p_{im}\ge t_{im}\}$. La exhaustividad es $\hat R_m=n_+^{-1}\sum_{i:y_i=1} Z_{im}$. Para la política del 98 %, $t_{im}$ es el cuantil de las probabilidades de los positivos OOF de los otros folds (misma regla discreta y comparación >= que el informe principal). No se fija el recall de la propia muestra evaluada por construcción. Sin embargo, las OOF de calibración pueden proceder de modelos entrenados con las filas del fold de evaluación; no es calibración plenamente anidada. En la sensibilidad el umbral es constante 0,50.

Se reporta la tasa de falsos positivos (FPR) junto con recall; maximizar recall sin controlar las falsas alarmas no constituye una política operativa óptima.


In [ ]:
politicas={'recall98_cruzado':None,'umbral050':.5}
resultados={}
for policy,t in politicas.items():
    Z,thresholds=decisions(y,P,folds,target=TARGET,fixed_threshold=t)
    thresholds.to_csv(OUT/f'{policy}_umbrales.csv',index=False)
    models,pairs,n_patterns=pairwise_recall(y,Z,names,B=B,seed=SEED,alpha=ALPHA)
    models=models.merge(ranking[['modelo','familia','roc_auc']],on='modelo',validate='one_to_one')
    assert len(pairs)==45
    models.to_csv(OUT/f'{policy}_recall.csv',index=False)
    pairs.to_csv(OUT/f'{policy}_pares.csv',index=False)
    resultados[policy]=(models,pairs,n_patterns)
    print(policy,'| patrones conjuntos de detección:',n_patterns)
    display(models[['modelo','familia','VP','FN','FP','recall','recall_ci_low','recall_ci_high','tasa_falsas_alarmas']].round(6))

# Optional verification against the original local report, if available.
original=ROOT/'outputs/informe_experiment024_es/metricas_alerta_cruzadas.csv'
if original.is_file():
    t=pd.read_csv(original).set_index('Código')
    m=resultados['recall98_cruzado'][0].set_index('familia')
    for c1,c2 in [('FN','FN'),('FP','FP'),('recall','Recall cruzado')]:
        if not np.allclose(m[c1],t.loc[m.index,c2],rtol=0,atol=5e-10):
            raise ValueError(f'Política 98 % no coincide con informe original: {c1}')
    print('Política cruzada verificada contra el informe original.')


## 3. Bootstrap pareado, McNemar y multiplicidad

Para cada par (fila $A$, columna $B$), $D_i=Z_{iA}-Z_{iB}\in\{-1,0,1\}$ y $\hat\Delta=(b-c)/n_+$, donde $b$ y $c$ cuentan los fraudes detectados solo por A o solo por B, respectivamente.

**Bootstrap:** se remuestrean conjuntamente los vectores binarios de los diez modelos sobre positivos. Si un patrón aparece $n_k$ veces, sus frecuencias de remuestreo tienen distribución multinomial $\mathrm{Multinomial}(n_+;(n_k/n_+)_k)$. Esto es exactamente equivalente en distribución a remuestrear filas positivas con reemplazo y conserva el emparejamiento de todos los pares. Se usan 100 000 réplicas, IC percentiles puntuales del 95 % y p bilateral aproximado centrado en el nulo:
$$p_{boot}=\frac{1+\sum_{r=1}^{B}\mathbf{1}\{|\Delta_r^*-\hat\Delta|\ge|\hat\Delta|\}}{B+1}.$$
No se interpreta la proporción bootstrap de diferencias positivas como un p-valor. Los umbrales y modelos permanecen fijos durante el remuestreo; no se captura su incertidumbre de entrenamiento/calibración.

**McNemar exacto:** bajo $H_0:R_A=R_B$, y suponiendo pares independientes, $b\mid(b+c)\sim\mathrm{Binomial}(b+c,1/2)$. Se aplica el contraste bilateral exacto (si no hay discordancias, $p=1$). Es preferible al t-test pareado para datos discretos y pares con pocas discordancias.

**Correcciones:** para $m=45$, $p_i^{Bonf}=\min(1,mp_i)$. Ordenando $p_{(1)}\le \cdots\le p_{(m)}$, $p_{(k)}^{Holm}=\min(1,\max_{j\le k}(m-j+1)p_{(j)})$. Holm controla el error familiar bajo dependencia arbitraria si los p-valores individuales son válidos; el bootstrap centrado es aproximado. Los IC puntuales no son simultáneos. Referencias: Holm (1979), *Scandinavian Journal of Statistics*, 6, 65–70; Westfall, Troendle y Pennello (2010), [Multiple McNemar Tests](https://pmc.ncbi.nlm.nih.gov/articles/PMC2902578/).


In [ ]:
for policy,(models,pairs,_) in resultados.items():
    print(policy)
    display(pairs[['fila','columna','solo_fila','solo_columna','delta_pp','ci_low','ci_high',
                   'p_mcnemar','p_mcnemar_holm','p_mcnemar_bonferroni',
                   'p_bootstrap','p_bootstrap_holm','p_bootstrap_bonferroni']].sort_values('p_mcnemar_holm').style.format({
                       'delta_pp':'{:+.4f}','ci_low':'{:+.6f}','ci_high':'{:+.6f}',
                       **{c:'{:.3g}' for c in pairs if c.startswith('p_')}}))


## 4. Mapas de calor triangulares inferiores

Cada celda corresponde a un par único; la diagonal y el triángulo superior se omiten. Las etiquetas M01–M10 corresponden al ranking AUC anterior. Se anotan los p-valores, y el color indica si son menores o iguales a 0,05 (sin ajuste, Holm o Bonferroni según el panel). No se representa ausencia de significación como equivalencia. Se exportan matrices numéricas y PNG/PDF para la presentación.


In [ ]:
def plot_pvalues(pairs,policy,method):
    columns=[f'p_{method}',f'p_{method}_holm',f'p_{method}_bonferroni']
    titles=['Sin ajuste','Holm (45 pares)','Bonferroni (45 pares)']
    fig,axes=plt.subplots(1,3,figsize=(21,7),constrained_layout=True)
    mask=np.triu(np.ones((len(names),len(names)),dtype=bool))
    for ax,col,title in zip(axes,columns,titles):
        M=p_matrix(pairs,names,col)
        M.to_csv(OUT/f'{policy}_{col}_matriz.csv')
        annotation=M.map(lambda p: '' if pd.isna(p) else ('<1e-300' if p==0 else f'{p:.2g}'))
        sns.heatmap((M<=ALPHA).astype(int),mask=mask,annot=annotation,fmt='',
            cmap=ListedColormap(['#e6edf3','#adcfbb']),vmin=0,vmax=1,square=True,
            linewidths=.6,linecolor='white',cbar=False,ax=ax,annot_kws={'size':9})
        ax.set_title(title,fontweight='bold')
        ax.set(xlabel='Modelo B (columna)',ylabel='Modelo A (fila)')
        ax.tick_params(axis='both',rotation=0)
    fig.suptitle(f'{policy} · '+('McNemar exacto (principal)' if method=='mcnemar' else 'Bootstrap pareado centrado (aproximado)'),fontsize=17,fontweight='bold')
    fig.legend(handles=[Patch(facecolor='#adcfbb',label='p ≤ 0,05'),Patch(facecolor='#e6edf3',label='p > 0,05')],
               loc='outside lower center',ncol=2,frameon=False)
    for suffix in ('png','pdf'):
        fig.savefig(OUT/f'{policy}_{method}_heatmap.{suffix}',dpi=220,bbox_inches='tight')
    plt.show()
    plt.close(fig)

for policy,(_,pairs,_) in resultados.items():
    for method in ('mcnemar','bootstrap'):
        plot_pvalues(pairs,policy,method)


## 5. Magnitud y dirección de las diferencias

Un valor positivo indica mayor recall del modelo de la fila respecto al de la columna, en puntos porcentuales. Los asteriscos identifican diferencias con p de McNemar-Holm ≤ 0,05. Las barras muestran intervalos bootstrap puntuales del 95 % frente a M01 (ganador nominal por AUC, no necesariamente por recall).


In [ ]:
for policy,(models,pairs,_) in resultados.items():
    M=p_matrix(pairs,names,'delta_pp')
    H=p_matrix(pairs,names,'p_mcnemar_holm')
    M.to_csv(OUT/f'{policy}_delta_pp_matriz.csv')
    lim=max(float(np.nanmax(np.abs(M.to_numpy()))),.0001)
    annotations=pd.DataFrame('',index=names,columns=names)
    for i in range(1,len(names)):
        for j in range(i):
            annotations.iloc[i,j]=f'{M.iloc[i,j]:+.3f}'+('*' if H.iloc[i,j]<=ALPHA else '')
    fig,ax=plt.subplots(figsize=(9,7))
    sns.heatmap(M,mask=np.triu(np.ones(M.shape,dtype=bool)),annot=annotations,fmt='',cmap='vlag',
                center=0,vmin=-lim,vmax=lim,square=True,ax=ax,cbar_kws={'label':'Diferencia de recall (pp)'})
    ax.set(title=f'{policy} · Δ recall (fila − columna)',xlabel='Modelo B',ylabel='Modelo A')
    fig.tight_layout()
    fig.savefig(OUT/f'{policy}_delta_heatmap.png',dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)

    q=pairs[pairs.columna==names[0]].copy()
    pos=np.arange(len(q))
    fig,ax=plt.subplots(figsize=(10,5.8))
    ax.hlines(pos,100*q.ci_low,100*q.ci_high,color='#446c87',lw=2)
    ax.scatter(q.delta_pp,pos,color='#173f5f',zorder=3)
    ax.axvline(0,ls='--',color='gray')
    ax.set(yticks=pos,yticklabels=q.fila,xlabel='Diferencia frente a M01 (pp)',title=f'{policy} · IC bootstrap 95 % (puntuales)')
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(OUT/f'{policy}_intervalos_vs_M01.png',dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)


## 6. Resumen y límites de interpretación

La hipótesis principal es igualdad de recall, no igualdad de AUC; la selección original se mantiene por AUC. Holm controla multiplicidad de los 45 pares de la política principal, no el sesgo de selección de los 40 candidatos. La sensibilidad 0,50 se considera descriptiva por separado. Las diferencias de recall deben interpretarse junto con FPR y magnitud. No se concluye equivalencia a partir de p > 0,05; para ello se necesitaría un margen de equivalencia preespecificado. Se guardan únicamente resultados agregados; las predicciones originales y los datos de Kaggle permanecen privados.


In [ ]:
summary=[]
for policy,(models,pairs,n_patterns) in resultados.items():
    winner=models.sort_values(['recall','tasa_falsas_alarmas'],ascending=[False,True]).iloc[0]
    summary.append(dict(politica=policy,n_positivos=int(y.sum()),n_pares=len(pairs),n_patrones=n_patterns,
        mayor_recall=winner.modelo,recall=float(winner.recall),fpr=float(winner.tasa_falsas_alarmas),
        mcnemar_sin_ajuste=int((pairs.p_mcnemar<=ALPHA).sum()),
        mcnemar_holm=int((pairs.p_mcnemar_holm<=ALPHA).sum()),
        mcnemar_bonferroni=int((pairs.p_mcnemar_bonferroni<=ALPHA).sum()),
        bootstrap_holm=int((pairs.p_bootstrap_holm<=ALPHA).sum())))
summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT/'resumen_inferencia.csv',index=False)
meta=dict(experiment='experiment027',source='experiment024',source_signature=manifest['firma'],
    n_models=len(names),n_rows=len(y),n_positive=int(y.sum()),B=B,seed=SEED,alpha=ALPHA,target_recall=TARGET,
    primary_policy='recall98_cruzado',primary_test='exact two-sided McNemar, Holm over 45 pairs',
    bootstrap='joint binary pattern multinomial, percentile CI, centered p-value',
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__),
    scope='Exploratory conditional on selected OOF models and fixed thresholds; no training uncertainty.')
(OUT/'manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf8')
print('Resultados agregados:',OUT)
